# 🧪 BioCirv AI Analysis Playground

Welcome to the BioCirv AI analysis environment. This notebook allows you to explore the biocirv project data using natural language queries powered by PandasAI and the CBORG LLM gateway.

## 🚀 Getting Started

### 1. Initialize Environment
Run the cell below to set up the connection to GCP and initialize the AI agent. This cell handles dependency installation, repository cloning, and authentication.

In [ ]:
# 1. Set Staging Environment Defaults
import os
import sys
import importlib

os.environ['INSTANCE_CONNECTION_NAME'] = 'biocirv-470318:us-west1:biocirv-staging'
os.environ['DB_IAM_USER'] = 'biocirv-staging-cr-worker@biocirv-470318.iam'
os.environ['DB_NAME'] = 'biocirv-staging'
os.environ['CLOUD_MODE'] = 'true'

print("🌐 Cloning repository...")
repo_path = '/content/biocirv-ai'
if os.path.exists(repo_path):
    !rm -rf {repo_path}
!git clone -b dev https://github.com/petercarbsmith/biocirv-ai.git -q

print("📦 Installing dependencies from manifest (this may take a minute)...")
# Install the repository itself to ensure all version pins (pandasai 3.0, scipy, etc.) are respected
# Using --pre to allow pandasai 3.0.0b20+ which supports Pandas 2.x
!pip install --pre -e {repo_path} pg8000 cloud-sql-python-connector -q

# 2. Configure Python Path
src_path = os.path.join(repo_path, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# 3. Run Initialization Modules
from ca_biositing.ai_exploration.colab_setup import setup_colab
setup_colab()

from ca_biositing.ai_exploration.sandbox_setup import init_sandbox, get_agent

# 4. Initialize Sandbox & Agent
llm, db_config = init_sandbox(cloud_mode=True)
agent = get_agent(llm, db_config)

print("\n✅ BioCirv AI Agent Ready!")

### 🛠️ Pre-flight Check
If you encounter issues, run this cell to verify the environment solve and module accessibility.

In [ ]:
# 1. Verify Dependencies
print("🔍 Verifying AI Stack...")
try:
    import pandasai
    import pandasai_sql
    from pandasai_sql import PostgreSQLConnector
    import pg8000
    from google.cloud.sql.connector import Connector
    print(f"✅ PandasAI: {pandasai.__version__}")
    print(f"✅ SQL Connector: Found")
    print(f"✅ GCP Connector: Found")
except ImportError as e:
    print(f"❌ Missing Module: {e}")
    print("Please re-run the initialization cell above.")

# 2. Verify Repo Path
import ca_biositing
print(f"✅ ca_biositing module loaded from: {ca_biositing.__file__}")

## 🔍 Starter Queries

Try running some of these queries to see the 'Trinity' output (Code, Data, Plot).

In [ ]:
# Query 1: Data Summary
result = agent.chat("Show me a summary of the available views in the ca_biositing schema.")
result.display()

In [ ]:
# Query 2: Visualization
result = agent.chat("Create a bar chart of the top 10 counties by biomass potential.")
result.display()

In [ ]:
# Query 3: Complex Analysis
result = agent.chat("Which counties have both high biomass potential and are within 50 miles of a major highway? Show the top 5.")
result.display()

## 🛠️ Advanced Usage

You can inspect the generated SQL and Python code for any query by looking at the `code` attribute of the result.